In [ ]:
# Run this cell first; launch Jupyter from this project or one of its subfolders.
from pathlib import Path
import os
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "cmit_utils.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Launch Jupyter from the CMIT project directory.")
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT / "Exercise")


In [ ]:
import math  as mth
import numpy as np
import pickle
import matplotlib.pyplot as plt

def Save(obj, doc_path = '.', save_name = 'pkl_file'):
    model_name = save_name + '.pkl' 
    with open(doc_path+ '/'+ model_name, 'wb') as file:
        pickle.dump(obj, file)     

def Read_pickle(doc_path):
        with open(doc_path, 'rb') as file:
            result = pickle.load(file)  
        return result

# Exercise 01

## Fibonacci Series

In [ ]:
# Fibonacci Series
fibSer , ratioSer = [1,1] , [1]

for i in range(2,20):
    an = fibSer[i-1] + fibSer[i-2]
    ratio = fibSer[i-1] / an
    fibSer.append(an)
    ratioSer.append(ratio)
    
print(fibSer)
print(ratioSer) # The ratio converge to the golden ratio

## Elliptic integral

In [ ]:
# Elliptic integral
def F(theta, k):
    x = (k*np.sin(theta))**2
    y = (1-x)**(-0.5) 
    return y

def K(k):
# Trapezium Method for integral
    theta   = np.linspace(0, np.pi/2, num=500)
    dtheta  = theta[1] - theta[0]
    F_value = F(theta, k)
    I = ( F_value.sum() - (F_value[0] + F_value[-1])/2 ) * dtheta 
    return I

# Calculate K(k) for k from 0 to 0.999, plot K(k) and save as a txt file
ks = np.linspace(0, 0.999, num=100)
Ks = [K(k) for k in ks]

# Plot
fig = plt.figure( figsize = (5,5) );
plt.plot(ks,Ks);
plt.xlabel('k'); 
plt.ylabel('K(k)');
plt.grid( visible = True );
fig.savefig('./Exercise01_EllipticIntegral.jpg', bbox_inches = 'tight')

# Save txt
f = open("./Exercise01_EllipticIntegral_writefiles.txt","w")
for i in range(len(ks)):
    writtenStr = '\tk = %.6f\tK(k) = %.6f\n'%(ks[i],Ks[i])
    f.write(writtenStr)
f.close()

## Matrix similarity

In [ ]:
from numpy import linalg as LA

In [ ]:
def similar(A,B):
# Check whether two matrices A and B are similar
    eigVal_A , _ = LA.eig(A)
    eigVal_B , _ = LA.eig(B)
    eigVal_A , eigVal_B = np.sort(eigVal_A) , np.sort(eigVal_B) # Sort eigenvalues
    
    if len(eigVal_A) != len(eigVal_B):
        return False
    else:
        return np.all( eigVal_A == eigVal_B )

In [ ]:
A = np.array([[0,1],[5,3]])
B = np.array([[1,2],[4,3]])
print(similar(A,B))

In [ ]:
C = np.array([[1,2],[-1,4]])
D = np.array([[-1,6],[-2,6]])
print(similar(C,D))

# Exercise 02

## Reverse of List

In [ ]:
# Random digits: list and reversed list
randlst = np.random.rand(10)
print(randlst)

randlst_reverse = randlst[::-1]
print(randlst_reverse)

# Reverse of string
Str = 'tacocat'
Str_reverse = Str[::-1]
print( Str == Str_reverse )

In [ ]:
# Sort reversely
f = open("Exercise02_GradeBook.txt","r")
Alpha = list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXY")
FinalGrade = {}

for line in f:
    
    line = line.replace("\n","").split(",")
    
    # If the grade starts with letters, drop it
    if line[1][0] in Alpha:
        continue
        
    FinalGrade.update( { line[0] : int(line[1])*0.1 + int(line[2])*0.9 } )
    
f.close()

# Print the list of tuples for the final grades
FinalGrade_descending = sorted(FinalGrade.items(), key = lambda x: x[1], reverse = True)
print(FinalGrade_descending, end = '\n\n')

# Print the highest 5 students' number
rank = 0
current_grade = 0

for grade in FinalGrade_descending:
          
    if grade[1] != current_grade:
        rank += 1
        current_grade = grade[1]
        
    if rank >= 6:
        break
    else:
        print(grade[0] + ", Final Grade %.1f"%grade[1])  

## Translation from Imread to Matplotlib

In [ ]:
import cv2
img = cv2.imread('Exercise02_France.png', cv2.IMREAD_COLOR)

plt.subplot(121) # Plot before reversion
plt.imshow(img);
plt.subplot(122) # Plot after reversion
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB));

In [ ]:
# Anoter way for reversion
img_RGB = img[:,:,[2,1,0]]
plt.imshow(img_RGB);

# Exercise 03 FNN & CNN

## Run an image classification model in the CIFAR10 dataset:
You will need to:
- Load the data, and load into dataloaders
- Define a network (see end)
- Define an optimizer (recommened [Stochastic Gradient Descent](https://pytorch.org/docs/stable/generated/torch.optim.SGD.html) with lr = 0.01)
- Define loss function (reccommend [CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) (note, a softmax function should not be used in the final layer of the network when using CrossEntropy))
- Run a training loop
- Write some code which print a montage of images, with the title on each subplot as the ground truth label and the predicted label

Some remarks:
- The images in CIFAR10 are RGB images, and are therefore of shape $(32,32,3)$, unlike MNIST which were greyscale and of shape $(28,28)$.
- Pytorch requires inputs to a convolutional layer to be of shape (BatchSize,Channels,Height,Width). When using dataloaders this is done automatically, and so during your training loop you won't need to worry about this. However, if defining your own function (for example for plotting the output of the model), executing: "img, label = train_data[0]" will error as img is of shape (3,32,32), and not (1,3,32,32) -- (as you are essentially passing a batch of 1 when predicting a single image **img.unsqueeze(0)**.
- A similar problem here is when plotting the image. matplotlib expects your image to be of shape (32,32,3) (i.e. the channels to be the final dim). To transpose an image from shape (3,32,32) to (32,32,3), use the np.transpose function as in: img = np.transpose(img,[1,2,0])


## Network:
You can in principle define your own architecture, but I recommend making a simple network according to the following pseduocode
* Conv2D (with filters 6, padding = 1) (remark: in_channels will be 3 here as RGB image!)
* ReLU, MaxPool
* Conv2D (with filters 16, padding = 1) 
* ReLU, MaxPool
* Flatten
* FullyConnectedLayer, 120 (has input 8x8x16)
* FullyConnectedLayer, 84
* FullyConnectedLayer, 10

As we aren't using softmax, the predicted class of the network will be the entry with the maximum value.

This network contains a relatively low number of parameters. This is so it will train in a reasonable amount of time on your machine. As a result, your network accuracy will not be amazing. Do not worry about this.

In [ ]:
# 数据准备用包
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader

# 搭建、训练网络用包
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable

In [ ]:
# load data
train_data = datasets.CIFAR10(root='data', train=True, transform=ToTensor(), download=True)
test_data = datasets.CIFAR10(root='data', train=False, transform=ToTensor())

# loaders
loaders = { 'train' : torch.utils.data.DataLoader(train_data, batch_size=50, shuffle=True, num_workers=1),
            'test'  : torch.utils.data.DataLoader(test_data,  batch_size=50, shuffle=True, num_workers=1) }

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [ ]:
# network construction
class CNN(nn.Module): 
    
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Sequential(   
            nn.Conv2d(in_channels=3, out_channels=6, kernel_size=(3,3), padding=1), # [BS,6,32,32]
            nn.ReLU(), 
            nn.MaxPool2d(kernel_size=(2,2)) # [BS,6,16,16]
        )
        
        self.conv2 = nn.Sequential(   
            nn.Conv2d(in_channels=6, out_channels=16, kernel_size=(3,3), padding=1), # [BS,16,16,16]
            nn.ReLU(), 
            nn.MaxPool2d(kernel_size=(2,2)) # [BS,16,8,8]
        )
        
            
        self.FC = nn.Sequential( 
            nn.Linear(1024, 120), # [BS, 16*8*8] -> [BS, 120]
            nn.Linear(120, 84),   # [BS, 120] -> [BS, 84]
            nn.Linear(84, 10)     # [BS, 84] -> [BS, 10]
        )
        
    def forward(self, x):        # [BS,3,32,32] -> [BS, 10]
        x = self.conv1(x) 
        x = self.conv2(x) 
        x = x.view(x.size(0), -1) # flatten 
        x = self.FC(x)            
        return x  

In [ ]:
# 训练函数
def train(model, loaders, num_epochs, learning_rate=0.01):
    
    ## 将模型设置为训练模式
    model.train() 
        
    # Optimizer & Loss function
    optimizer = optim.SGD(model.parameters(), lr = learning_rate)   # 采用Adam优化器
    loss_func = nn.CrossEntropyLoss()   
    
    for epoch in range(num_epochs):
        for i, (features, labels) in enumerate(loaders['train']):   
            
            ## Forward calculation for the loss function 
            output = model(features)        # model.forward(features)
            loss = loss_func(output, labels)
 
            ## Backpropagation, update parameters
            optimizer.zero_grad()           # clear gradients
            loss.backward()                 # back propgation 
            optimizer.step()                # update parameters
            
            # Display the training progress
            if (i+1) % 100 == 0:
                print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}' 
                       .format(epoch + 1, num_epochs, i + 1, len(loaders['train']), loss.item())) 
                
                
# 测试函数                
def test(model):
    
    ## 将模型设置为测试模式
    model.eval()
    
    test_loss, correct = 0, 0 #  累计测试集上的总损失和正确预测的样本数
    
    with torch.no_grad():     ## 这是一个上下文管理器，用于暂时关闭自动求导
        
        for features, labels in loaders['test']:
            output = model(features)
            test_loss += F.cross_entropy(output, labels, reduction='sum').item() # 计算loss
            pred = output.data.max(1, keepdim=True)[1]          # 返回值是一个元组，第一个元素是最大值tensor，第二个元素是最大值所在的索引tensor。    
            correct += pred.eq(labels.data.view_as(pred)).sum() # 统计正确个数
            
        test_loss /= len(loaders['test'].dataset)               # loaders['test'].dataset 获得所有测试样本
        
        print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
          test_loss, correct, len(loaders['test'].dataset), 100. * correct / len(loaders['test'].dataset)))

In [ ]:
# model = CNN()
train(model, loaders, num_epochs=100, learning_rate=0.003)
test(model)

In [ ]:
# Save Model
torch.save(model.state_dict(), './Exercise03_CNNmodel.pth')

# Load Model
model = CNN()  # 这是你的自定义模型类
model.load_state_dict(torch.load('./Exercise03_CNNmodel.pth', map_location="cpu", weights_only=True))

In [ ]:
# Plot multiple
figure = plt.figure(figsize=(15, 12))
cols, rows = 5, 5
for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(test_data), size=(1,)).item()
    img, label = test_data[sample_idx]
    output = model(img.unsqueeze(0))
    pred_label = torch.max(output, 1)[1].numpy() # .numpy()将Tensor对象转换为NumPy ndarray对象
    figure.add_subplot(rows, cols, i)
    plt.title('gt = ' + classes[label] + ', pred = '+ classes[pred_label.item()])
    plt.axis("off")
    plt.imshow(np.transpose(img, [1,2,0]), cmap="gray")
plt.show()

# Exercise 04 Image Processing
- Try training the segmentation problem again with DICE loss function.
- The DICE coefficient is a measure of how well two sets coincide with one another, i.e. how well two segmentation results are similar. We use the output of our network, $u$, and the ground truth, $GT$, to measure performance.

$$ DICE\ (\ u,\ GT\ ) = \frac{ 2 \cdot | u \cdot GT |}{|u| + |GT|} $$

<p align="center">
<img src="../Notes/Notes_IMProcess/dice.png" width="250" title="DICE." >
</p> 

- $|u|=Sum(u)$, same for $|GT|$
- A score of 1 is a perfect match, whereas a score of 0 is the opposite. We want a score of 1 in segmentation.
- To use this in a loss function, the loss function looks like:

$$ DICELOSS\ (\ u,\ GT\ ) = 1 - DICE\ (\ u,\ GT\ ) $$

In [ ]:
# 数据准备用包
import torch
import torchvision.transforms as T
from torchvision import datasets
from torch.utils.data import DataLoader

# 搭建、训练网络用包
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable

# 其他数据处理包
from skimage import io, transform
from PIL import Image

## 读入数据

In [ ]:
from cmit_utils import get_paths


In [ ]:
from cmit_utils import SegmentationDataset as CustomDataset


In [ ]:
# 数据准备
train_paths, test_paths = get_paths()
train_data = CustomDataset(train_paths)
test_data  = CustomDataset(test_paths)

# 构造loaders
loaders = { 'train' : torch.utils.data.DataLoader(dataset=train_data, batch_size=4, shuffle=True),
            'test'  : torch.utils.data.DataLoader(dataset=test_data, batch_size=4, shuffle=True) }

# 检查维度
for images , labels in loaders['train']:
    print(images.size())
    print(labels.size(), end='\n\n')

## 构建 CNN 网络与损失函数 DICELOSS

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(DiceLoss, self).__init__()

    def forward(self, inputs, targets, smooth=1):
        
        ### comment out if your model contains a sigmoid or equivalent activation layer
        # inputs = F.sigmoid(inputs)       
        
        ### flatten target and prediction tensors
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        
        intersection = (inputs * targets).sum()                            
        dice = (2.*intersection + smooth)/(inputs.sum() + targets.sum() + smooth)  
        
        return 1 - dice

In [ ]:
class CNN(nn.Module):
    def __init__(self, img_size, in_channels=3, out_channels=1):
        super().__init__()
        
        f = [ 10, 20, 30, 20, 10, out_channels ]
        
        self.conv1 = nn.Sequential(   
            nn.Conv2d(in_channels=in_channels, out_channels=f[0], kernel_size=(3,3), padding=1), 
            nn.ReLU(), 
            nn.MaxPool2d(kernel_size=(2,2)) 
        )
        self.conv2 = nn.Sequential( 
            nn.Conv2d(in_channels=f[0], out_channels=f[1], kernel_size=(3,3), padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,2)) 
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(in_channels=f[1], out_channels=f[2], kernel_size=(3,3), padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=f[2], out_channels=f[2], kernel_size=(3,3), padding=1),
            nn.ReLU()
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(in_channels=f[2], out_channels=f[3], kernel_size=(3,3), padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2))
        
        self.conv5 = nn.Sequential(
            nn.Conv2d(in_channels=f[3], out_channels=f[4], kernel_size=(3,3), padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2))
        
        self.conv6 = nn.Sequential(
            nn.Conv2d(in_channels=f[4], out_channels=f[5], kernel_size=(1,1)),
            nn.Sigmoid()
        )
            

    def forward(self, x):

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        x = self.conv6(x)
        
        return x

In [ ]:
# 训练函数
def train(model, loaders, num_epochs=100, learning_rate=0.001):
    
    ## 将模型设置为训练模式
    model.train() 
        
    # Optimizer & Loss function
    optimizer = optim.Adam(model.parameters(), lr = learning_rate)   # 采用Adam优化器
    loss_func = DiceLoss() ##
    
    for epoch in range(num_epochs):
        for i, (features, labels) in enumerate(loaders['train']):  
            
            ## Forward calculation for the loss function 
            output = model(features)        
            loss = loss_func(output, labels)
 
            ## Backpropagation, update parameters
            optimizer.zero_grad()           
            loss.backward()                 
            optimizer.step()                
            
            # Display the training progress
            if (i+1) % 2 == 0:
                print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}' 
                       .format(epoch + 1, num_epochs, i + 1, len(loaders['train']), loss.item()))   

## 训练与可视化

In [ ]:
# Training
CNN_model = CNN(img_size=256, in_channels=3, out_channels=1)
train(CNN_model, loaders, num_epochs=300, learning_rate=0.001)

In [ ]:
# Save & Load Model
# torch.save(CNN_model.state_dict(), './Exercise04_CNNSeg.pth')
CNN_model = CNN(img_size=256, in_channels=3, out_channels=1)  
CNN_model.load_state_dict(torch.load('./Exercise04_CNNSeg.pth', map_location="cpu", weights_only=True))

In [ ]:
dataset = test_data
train_loader = loaders['test']

with torch.no_grad():
    figure = plt.figure(figsize=(10, 8))
    rows = 3
    for i in range(0, rows):
        sample_idx = torch.randint(len(dataset), size=(1,)).item()
        img, label = dataset[sample_idx]

        CNN_output  = CNN_model(img.unsqueeze(0))

        figure.add_subplot(rows, 3, 3*i + 1)
        plt.axis("off")
        plt.imshow(img.permute(1,2,0))
        plt.title("image")
        
        figure.add_subplot(rows, 3, 3*i + 2)
        plt.axis("off")
        plt.imshow(CNN_output.squeeze(), cmap="gray")
        plt.title("CNN, epoch 300, DICELOSS")
        
        figure.add_subplot(rows, 3, 3*i + 3)
        plt.axis("off")
        plt.imshow(torch.squeeze(label), cmap="gray")
        plt.title("ground truth (target label)")